# Практическое занятие 11. Очистка и преобразование больших объёмов данных с использованием библиотеки Pandas

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/p11.ipynb)

**Цели занятия:**

- Находить и исправлять типичные проблемы данных: дубликаты, пропуски, разные написания, неверные типы, выбросы
- Приводить к единому виду строки, числа, даты и телефоны с помощью методов `str`, `to_numeric`, `to_datetime`
- Экономить память (тип `category`, понижение разрядности) и обрабатывать большие файлы по частям (`chunksize`)
- Агрегировать очищенные данные: `groupby`, `transform`, `pivot_table`

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/OWNER/REPO.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Зачем чистить данные

В реальных проектах на очистку и подготовку данных уходит до 80 % времени. «Грязные» данные дают неверные выводы: если Ташкент записан шестью способами (`Toshkent`, `toshkent`, `Toshkent sh.`, `Tashkent`, `TOSHKENT`, `" Toshkent "`), то в статистике по городам он «развалится» на шесть разных городов.

Файл `data/students_dirty.csv` содержит все типичные проблемы:

| Проблема | Пример | Как исправить в pandas |
|---|---|---|
| Дубликаты строк | один студент записан дважды | `df.duplicated()`, `df.drop_duplicates()` |
| Лишние пробелы, регистр | `"  MURODOV BEHRUZ "` | `str.strip()`, `str.split()`, `str.capitalize()` |
| Разные написания | `Samarkand` / `samarqand` | `str.lower()` + словарь замен `replace({...})` |
| Разные апострофы | `Farg‘ona` / `Farg'ona` | `str.replace("‘", "'")` |
| Число как строка с запятой | `"3,9"` | `str.replace(",", ".")` + `pd.to_numeric` |
| Пропуски | пустой GPA | `fillna`, `groupby(...).transform("median")` |
| Выбросы | возраст `150`, `-1` | `between`, `clip`, пересчёт из `birth_year` |
| Даты в разных форматах | `2023-09-01`, `01.09.2023`, `2023/09/01` | `pd.to_datetime(..., format="mixed", dayfirst=True)` |
| Телефоны в разном виде | `+998963922256` | регулярные выражения `str.replace(r"\D", "", regex=True)` |

## Строки: методы .str

```python
df["city"] = (df["city"].str.strip()            # убрать пробелы по краям
                        .str.replace("‘", "'")   # единый апостроф
                        .str.lower()             # единый регистр
                        .replace({"tashkent": "toshkent", "toshkent sh.": "toshkent"})
                        .str.capitalize())       # "toshkent" -> "Toshkent"
```

> **Внимание: str.title() и узбекские имена**  
> `"QODIROV ULUG'BEK".title()` даёт `"Qodirov Ulug'Bek"`: после апострофа Python считает, что началось новое слово. Для узбекских имён надёжнее делать заглавной только первую букву каждого слова: `" ".join(w.capitalize() for w in name.split())`. Это та же задача **нормализации текста**, что и в компьютерной лингвистике (oʻ, gʻ, ʼ).

## Числа, пропуски и выбросы

```python
df["gpa"] = pd.to_numeric(df["gpa"].str.replace(",", "."), errors="coerce")   # плохие значения -> NaN
df["gpa"] = df["gpa"].fillna(df.groupby("faculty")["gpa"].transform("median"))  # медиана своего факультета
bad = ~df["age"].between(16, 60)                 # маска выбросов
df.loc[bad, "age"] = 2026 - df.loc[bad, "birth_year"]
```

Метод `transform` возвращает Series **той же длины**, что и исходная таблица: каждой строке — значение её группы. Поэтому его удобно использовать для заполнения пропусков и для вычисления отклонений от среднего по группе.

## Даты и телефоны

```python
df["enroll_date"] = pd.to_datetime(df["enroll_date"], format="mixed", dayfirst=True)
digits = df["phone"].str.replace(r"\D", "", regex=True)           # только цифры: 998901234567
df["phone"] = digits.str.replace(r"^998(\d{2})(\d{3})(\d{2})(\d{2})$", r"+998 \1 \2-\3-\4", regex=True)
```

## Память и большие файлы

Строки в pandas хранятся как объекты Python, и каждая занимает десятки байт. Если значений мало (4 факультета, 12 городов), выгоднее тип **category**: значения хранятся один раз, а в столбце — маленькие целые коды.

```python
df.memory_usage(deep=True).sum()                 # память в байтах
df["faculty"] = df["faculty"].astype("category")
df["course"] = df["course"].astype("int8")       # 1..4 помещается в 1 байт вместо 8
df["gpa"] = df["gpa"].astype("float32")

# типы можно задать сразу при чтении
big = pd.read_csv("big.csv", dtype={"faculty": "category", "course": "int8"}, usecols=["faculty", "course", "gpa"])
```

Если файл не помещается в память, его читают **по частям**: `chunksize` возвращает итератор DataFrame-ов. Каждую часть обрабатывают и сохраняют только агрегаты (суммы и количества), а в конце объединяют:

```python
parts = []
for chunk in pd.read_csv("big.csv", chunksize=50_000):
    parts.append(chunk.groupby("faculty")["gpa"].agg(["sum", "count"]))
total = pd.concat(parts).groupby(level=0).sum()
mean_gpa = total["sum"] / total["count"]          # среднее нельзя усреднять по частям — только сумма/количество!
```

> **Совет: Сводные таблицы**  
> `pd.pivot_table(df, values="gpa", index="faculty", columns="course", aggfunc="mean")` — средний GPA в разрезе «факультет × курс», аналог сводной таблицы Excel.

## Задание 1. Разведка: дубликаты и пропуски

1. Прочитайте `data/students_dirty.csv` в `raw` и осмотрите его (`info()`).
2. `n_dups` — число полностью повторяющихся строк; `missing` — Series с числом пропусков в каждом столбце; `n_cities_raw` — сколько **различных** написаний городов в столбце `city`.
3. `df` — таблица без дубликатов с заново пронумерованным индексом (`reset_index(drop=True)`).

<details><summary>💡 Подсказка</summary>

`raw.duplicated().sum()`, `raw.isna().sum()`, `raw["city"].nunique()`, `raw.drop_duplicates().reset_index(drop=True)`.

</details>

In [ ]:
import pandas as pd

raw = ...
raw.info()

n_dups = ...
missing = ...
n_cities_raw = ...

df = ...

print("Дубликатов:", n_dups, "| вариантов города:", n_cities_raw)
print(missing[missing > 0])
print(raw.shape, "->", df.shape)

## Задание 2. Нормализация строк: ФИО и города

В `df` (уже без дубликатов) приведите к единому виду:

1. `full_name` — без лишних пробелов, каждое слово с заглавной буквы: `"  MURODOV BEHRUZ "` → `"Murodov Behruz"`, а `"Qodirov Ulug'bek"` должно остаться как есть (осторожно со `str.title()`!).
2. `city` — одно каноническое написание для каждого города: `Toshkent`, `Samarqand`, `Farg'ona` (с обычным апострофом `'`) и т.д. Посмотрите все варианты: `df["city"].value_counts()`.
3. `n_cities` — число различных городов после очистки.

<details><summary>💡 Подсказка</summary>

ФИО: `df["full_name"].str.split().apply(lambda ws: " ".join(w.capitalize() for w in ws))`. Город: `.str.strip().str.replace("‘", "'").str.lower().replace(fix).str.capitalize()`; в `fix` нужны `"toshkent sh."`, `"samarkand"`, `"fargona"`.

</details>

In [ ]:
# Подготовительный код задания
import pandas as pd
df = pd.read_csv("data/students_dirty.csv").drop_duplicates().reset_index(drop=True)

In [ ]:
print(df["city"].value_counts())

df["full_name"] = ...

fix = {
    "tashkent": "toshkent",
    # допишите остальные варианты (в нижнем регистре)
}
df["city"] = ...

n_cities = df["city"].nunique()
print(df["city"].value_counts())
print("Городов:", n_cities)

## Задание 3. Числа: GPA, выбросы, пропуски

1. `gpa` записан строкой, иногда с запятой (`"3,9"`). Преобразуйте столбец в число (float).
2. `n_gpa_missing` — сколько GPA пропущено **после** преобразования; заполните пропуски **медианой GPA своего факультета** (`groupby` + `transform`).
3. В `age` есть невозможные значения. `n_bad_age` — число строк, где возраст вне диапазона 16–60; для них пересчитайте возраст как `2026 - birth_year`.
4. Пропуски в `attendance` заполните медианой всего столбца.

<details><summary>💡 Подсказка</summary>

`pd.to_numeric(df["gpa"].str.replace(",", "."))`; медиана группы: `df.groupby("faculty")["gpa"].transform("median")`; маска выбросов: `bad = ~df["age"].between(16, 60)`, затем `df.loc[bad, "age"] = 2026 - df.loc[bad, "birth_year"]`.

</details>

In [ ]:
# Подготовительный код задания
import pandas as pd
df = pd.read_csv("data/students_dirty.csv").drop_duplicates().reset_index(drop=True)

In [ ]:
df["gpa"] = ...
n_gpa_missing = ...
df["gpa"] = ...

bad = ...
n_bad_age = ...
# исправьте возраст в строках bad

df["attendance"] = ...

print("Пропусков GPA было:", n_gpa_missing, "| выбросов возраста:", n_bad_age)
df[["gpa", "age", "attendance"]].describe()

## Задание 4. Даты и телефоны

1. Столбец `enroll_date` содержит даты в трёх форматах: `2023-09-01`, `2023/09/01`, `01.09.2023` (день.месяц.год). Преобразуйте его в datetime. Все студенты поступили **1 сентября**.
2. Приведите все телефоны к виду `+998 90 123-45-67`. Сначала оставьте только цифры, затем отформатируйте регулярным выражением.
3. `n_changed` — у скольких студентов телефон изменился после нормализации.
4. `by_year` — Series: сколько студентов поступило в каждом году (`dt.year`), по возрастанию года.

<details><summary>💡 Подсказка</summary>

Даты: `pd.to_datetime(df["enroll_date"], format="mixed", dayfirst=True)`. Цифры: `df["phone"].str.replace(r"\D", "", regex=True)`. Формат: `digits.str.replace(r"^998(\d{2})(\d{3})(\d{2})(\d{2})$", r"+998 \1 \2-\3-\4", regex=True)`.

</details>

In [ ]:
# Подготовительный код задания
import pandas as pd
df = pd.read_csv("data/students_dirty.csv").drop_duplicates().reset_index(drop=True)

In [ ]:
old_phone = df["phone"].copy()

df["enroll_date"] = ...

digits = ...
df["phone"] = ...
n_changed = ...

by_year = ...

print(by_year)
print("Изменено телефонов:", n_changed)
df[["full_name", "enroll_date", "phone"]].head()

## Задание 5. Экономия памяти: category и downcast

Подготовительный код создал «большой» файл `data/big_students.csv` (300 000 студентов, путь — в переменной `BIG`).

1. Прочитайте его в `big` и посчитайте `mem_before` — память в **мегабайтах** (`memory_usage(deep=True).sum() / 1024**2`).
2. Создайте `big_opt` — копию с оптимальными типами: `faculty` и `city` → `category`, `course` → `int8`, `gpa` и `attendance` → `float32`, `student_id` → `int32`.
3. `mem_after` — память `big_opt` в МБ; `saving_pct` — экономия в процентах: `(1 - mem_after / mem_before) * 100`.

<details><summary>💡 Подсказка</summary>

`big.astype({"faculty": "category", "city": "category", "course": "int8", "gpa": "float32", "attendance": "float32", "student_id": "int32"})`.

</details>

In [ ]:
# Подготовительный код задания
import os
import numpy as np
import pandas as pd
BIG = "data/big_students.csv"
if not os.path.exists(BIG):
    _rng = np.random.default_rng(2026)
    _n = 300_000
    _fac = np.array(["Kompyuter lingvistikasi", "Filologiya", "Tarjimonlik", "Xalqaro jurnalistika"])
    _city = np.array(["Toshkent", "Samarqand", "Buxoro", "Andijon", "Namangan", "Farg'ona", "Nukus", "Qarshi"])
    _fi = _rng.integers(0, 4, _n)
    _gpa = np.round(np.clip(_rng.normal(3.9, 0.45, _n) + np.array([0.12, 0.0, -0.05, 0.04])[_fi], 2.0, 5.0), 2)
    _gpa[_rng.random(_n) < 0.02] = np.nan
    pd.DataFrame({
        "student_id": np.arange(1, _n + 1),
        "faculty": _fac[_fi],
        "city": _city[_rng.integers(0, 8, _n)],
        "course": _rng.integers(1, 5, _n),
        "gpa": _gpa,
        "attendance": np.round(_rng.uniform(50, 100, _n), 1),
    }).to_csv(BIG, index=False)
    del _rng, _n, _fac, _city, _fi, _gpa

In [ ]:
big = pd.read_csv(BIG)
print(big.dtypes)

mem_before = ...

big_opt = big.astype({
    "faculty": ...,
    # допишите остальные столбцы
})

mem_after = ...
saving_pct = ...

print(big_opt.dtypes)
print(f"{mem_before:.1f} МБ -> {mem_after:.1f} МБ, экономия {saving_pct:.0f}%")

## Задание 6. Чтение большого файла по частям

Представим, что `data/big_students.csv` (`BIG`) не помещается в память. Обработайте его **по частям** по 50 000 строк, читая только нужные столбцы (`usecols`):

- `n_chunks` — сколько частей прочитано; `total_rows` — сколько всего строк;
- `mean_gpa` — Series: средний GPA по факультетам (пропуски GPA не учитываются). Для каждой части сохраняйте **сумму и количество**, а среднее вычислите в конце;
- `low_att` — сколько студентов с посещаемостью ниже 60 %;
- `best_faculty` — факультет с наибольшим средним GPA.

<details><summary>💡 Подсказка</summary>

В цикле: `n_chunks += 1`, `total_rows += len(chunk)`, `low_att += (chunk["attendance"] < 60).sum()`, `parts.append(chunk.groupby("faculty")["gpa"].agg(["sum", "count"]))`. После цикла: `total = pd.concat(parts).groupby(level=0).sum()`, `mean_gpa = total["sum"] / total["count"]`.

</details>

In [ ]:
# Подготовительный код задания
import os
import numpy as np
import pandas as pd
BIG = "data/big_students.csv"
if not os.path.exists(BIG):
    _rng = np.random.default_rng(2026)
    _n = 300_000
    _fac = np.array(["Kompyuter lingvistikasi", "Filologiya", "Tarjimonlik", "Xalqaro jurnalistika"])
    _city = np.array(["Toshkent", "Samarqand", "Buxoro", "Andijon", "Namangan", "Farg'ona", "Nukus", "Qarshi"])
    _fi = _rng.integers(0, 4, _n)
    _gpa = np.round(np.clip(_rng.normal(3.9, 0.45, _n) + np.array([0.12, 0.0, -0.05, 0.04])[_fi], 2.0, 5.0), 2)
    _gpa[_rng.random(_n) < 0.02] = np.nan
    pd.DataFrame({
        "student_id": np.arange(1, _n + 1),
        "faculty": _fac[_fi],
        "city": _city[_rng.integers(0, 8, _n)],
        "course": _rng.integers(1, 5, _n),
        "gpa": _gpa,
        "attendance": np.round(_rng.uniform(50, 100, _n), 1),
    }).to_csv(BIG, index=False)
    del _rng, _n, _fac, _city, _fi, _gpa

In [ ]:
n_chunks = 0
total_rows = 0
low_att = 0
parts = []

for chunk in pd.read_csv(BIG, chunksize=..., usecols=["faculty", "gpa", "attendance"]):
    ...

total = ...
mean_gpa = ...
best_faculty = ...

print(n_chunks, "частей,", total_rows, "строк; посещаемость < 60%:", low_att)
print(mean_gpa.round(3))
print("Лучший факультет:", best_faculty)

## Задание 7. Агрегация чистых данных: transform и pivot_table

`df` — полностью очищенная таблица (см. подготовительный код). Проанализируйте её:

1. `df["gpa_diff"]` — отклонение GPA студента от **среднего GPA его группы** (`group`), используйте `transform`.
2. `df["rank_in_faculty"]` — место студента по GPA внутри факультета (1 — лучший; `groupby(...)["gpa"].rank(ascending=False, method="min")`).
3. `pivot` — сводная таблица среднего GPA: строки — `faculty`, столбцы — `course` (`pd.pivot_table`), округлите до 2 знаков.
4. `best_cell` — кортеж `(факультет, курс)` с максимальным средним GPA в `pivot`.
5. `leaders` — DataFrame лучших студентов каждого факультета (`rank_in_faculty == 1`) со столбцами `faculty`, `full_name`, `gpa`.

<details><summary>💡 Подсказка</summary>

`df.groupby("group")["gpa"].transform("mean")`; `pd.pivot_table(df, values="gpa", index="faculty", columns="course", aggfunc="mean").round(2)`; ячейка с максимумом: `pivot.stack().idxmax()`.

</details>

In [ ]:
# Подготовительный код задания
import pandas as pd
# полный конвейер очистки из заданий 1–4
df = pd.read_csv("data/students_dirty.csv").drop_duplicates().reset_index(drop=True)
df["full_name"] = df["full_name"].str.split().apply(lambda ws: " ".join(w.capitalize() for w in ws))
df["city"] = (df["city"].str.strip().str.replace("‘", "'").str.lower()
              .replace({"tashkent": "toshkent", "toshkent sh.": "toshkent", "samarkand": "samarqand", "fargona": "farg'ona"})
              .str.capitalize())
df["gpa"] = pd.to_numeric(df["gpa"].str.replace(",", "."))
df["gpa"] = df["gpa"].fillna(df.groupby("faculty")["gpa"].transform("median"))
_bad = ~df["age"].between(16, 60)
df.loc[_bad, "age"] = 2026 - df.loc[_bad, "birth_year"]
df["attendance"] = df["attendance"].fillna(df["attendance"].median())
df["enroll_date"] = pd.to_datetime(df["enroll_date"], format="mixed", dayfirst=True)
del _bad

In [ ]:
df["gpa_diff"] = ...
df["rank_in_faculty"] = ...

pivot = ...
best_cell = ...

leaders = ...

print("Лучшая комбинация:", best_cell)
print(leaders)
pivot

## Самостоятельная работа

## Самостоятельная работа: «Методы очистки больших объёмов данных» и «Сравнение библиотек NumPy и Dask»

1. Оформите конвейер очистки `students_dirty.csv` как функцию `clean_students(df)` из маленьких шагов и примените её через `df.pipe(...)`. Сохраните результат в CSV и Parquet, сравните размеры файлов.
2. Найдите выбросы GPA и посещаемости методом **межквартильного размаха** (IQR: значения вне `[Q1 − 1.5·IQR; Q3 + 1.5·IQR]`) и решите, удалять их или ограничивать (`clip`).
3. В Colab установите `dask` и сравните время вычисления среднего GPA по факультетам для файла на 5 млн строк тремя способами: pandas целиком, pandas с `chunksize`, `dask.dataframe`. Объясните, чем Dask отличается от NumPy (ленивые вычисления, граф задач, параллелизм).